# Structuring Documents for RAG: Document Loaders, Vector Stores, and Retrievers

## 1. Theory & Real-World Context

To build a Retrieval-Augmented Generation (RAG) pipeline, large language models need external, domain-specific context. LangChain orchestrates this through three distinct indexing and retrieval components:

* **Document Loaders (`S3FileLoader`):** The ingestion bridge. Loaders connect to external data sources (Amazon S3, databases, APIs, local files), extract the raw text, and convert it into standard LangChain `Document` objects (text + metadata).
* **Vector Stores (`OpenSearchVectorSearch`):** Specialized databases that store the high-dimensional embedding vectors generated from your documents. **Amazon OpenSearch Serverless (AOSS)** offers a dedicated Vector Engine collection type optimized for billion-scale cosine similarity searches without managing infrastructure. Another common AWS pattern is using Amazon Aurora PostgreSQL with the `pgvector` extension.
* **Retrievers (`AmazonKendraRetriever` / `vector_store.as_retriever()`):** The querying interface. When a user asks a question, the retriever converts the query into a vector, searches the vector store for the most mathematically similar documents, and returns them to the LLM.
* *Real-World AI Engineering:* **Amazon Kendra** is a fully managed enterprise search service. Unlike OpenSearch (where you must manage the embeddings, chunking, and vector database manually), Kendra has built-in ML ranking and natural language understanding. You point Kendra at an S3 bucket, and it handles the indexing and retrieval algorithms behind the scenes.



---

## 2. Visual Architecture

```mermaid
flowchart TD
    subgraph DataIngestion [Data Ingestion Pipeline]
        S3[Amazon S3 Bucket] -->|S3FileLoader| LOAD[Raw Documents]
        LOAD -->|Text Splitters| CHUNK[Document Chunks]
        CHUNK -->|BedrockEmbeddings| VS[(Amazon OpenSearch Serverless)]
    end

    subgraph QueryPipeline [Retrieval & Inference Pipeline]
        USER[User Query] --> EMBED[BedrockEmbeddings]
        EMBED -->|Cosine Similarity Search| VS
        VS -->|Top-K Documents| RET[LangChain Retriever]
        RET --> LCEL[LCEL Retrieval Chain]
        USER --> LCEL
        LCEL --> LLM[ChatBedrockConverse]
        LLM --> RESP[AI Response]
    end

```

---

## 3. Console Hands-On

**Objective:** Provision an Amazon OpenSearch Serverless Vector Engine to act as the backend for your LangChain Vector Store.

1. Navigate to the **Amazon OpenSearch Service Console** $\rightarrow$ Expand **Serverless** in the left menu $\rightarrow$ Click **Collections**.
2. Click **Create collection**.
3. **Collection name:** `ai-knowledge-base`.
4. **Collection type:** Select **Vector search** (optimized for ML embeddings and RAG).
5. **Security configuration:**
* *Encryption:* Select **AWS owned key** (or a Customer Managed KMS key for production).
* *Network access:* Select **Public** (for testing via local Python scripts) or **VPC** (for production EC2/Lambda access).
* *Data access:* Create a rule granting your IAM User/Role permission to `WriteDocument` and `ReadDocument` on all indexes within this collection.


6. Click **Submit**. Provisioning takes 3-5 minutes.
7. Once Active, copy the **Collection endpoint URL** (e.g., `[https://xxxx.us-east-1.aoss.amazonaws.com](https://xxxx.us-east-1.aoss.amazonaws.com)`).

---

## 4. Boto3 / LangChain Implementation

Production-grade script demonstrating how to load a document from S3, embed it into OpenSearch Serverless, and query it using modern LCEL (LangChain Expression Language) routing.

```python
import os
import logging
import boto3
from opensearchpy import RequestsHttpConnection
from requests_aws4auth import AWS4Auth
from botocore.exceptions import ClientError

from langchain_community.document_loaders import S3FileLoader
from langchain_aws import BedrockEmbeddings, ChatBedrockConverse
from langchain_community.vectorstores import OpenSearchVectorSearch
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("rag_pipeline")

class RAGPipelineManager:
    """Manages S3 loading, OpenSearch vector storage, and LCEL retrieval."""

    def __init__(self, aoss_endpoint: str, region_name: str = "us-east-1"):
        self.region = region_name
        self.aoss_endpoint = aoss_endpoint
        
        # Initialize standard boto3 clients
        self.boto3_session = boto3.Session()
        self.credentials = self.boto3_session.get_credentials()
        self.bedrock_client = self.boto3_session.client("bedrock-runtime", region_name=self.region)
        
        # AWS4Auth handles SigV4 signing required by OpenSearch Serverless
        self.awsauth = AWS4Auth(
            self.credentials.access_key,
            self.credentials.secret_key,
            self.region,
            "aoss",
            session_token=self.credentials.token
        )
        
        self.embeddings = BedrockEmbeddings(
            client=self.bedrock_client,
            model_id="amazon.titan-embed-text-v2:0"
        )
        
        self.llm = ChatBedrockConverse(
            client=self.bedrock_client,
            model="anthropic.claude-3-5-sonnet-20241022-v2:0",
            temperature=0.0 # Zero temperature for factual RAG
        )

    def load_and_index_s3_document(self, bucket: str, key: str, index_name: str) -> bool:
        """Loads a document from S3 and indexes its embeddings into OpenSearch Serverless."""
        try:
            logger.info(f"Loading s3://{bucket}/{key}...")
            loader = S3FileLoader(bucket, key)
            documents = loader.load()
            
            # Note: In production, insert a RecursiveCharacterTextSplitter here to chunk large docs
            
            logger.info(f"Indexing {len(documents)} document chunk(s) into AOSS index '{index_name}'...")
            OpenSearchVectorSearch.from_documents(
                documents=documents,
                embedding=self.embeddings,
                opensearch_url=self.aoss_endpoint,
                http_auth=self.awsauth,
                timeout=300,
                use_ssl=True,
                verify_certs=True,
                connection_class=RequestsHttpConnection,
                index_name=index_name,
                engine="faiss" # Standard vector engine type
            )
            return True
        except Exception as e:
            logger.error(f"Failed to load/index document: {str(e)}")
            return False

    def query_rag_pipeline(self, query: str, index_name: str) -> str:
        """Executes a modern LCEL RAG chain using the OpenSearch retriever."""
        # 1. Connect to existing OpenSearch index
        vector_store = OpenSearchVectorSearch(
            index_name=index_name,
            embedding_function=self.embeddings,
            opensearch_url=self.aoss_endpoint,
            http_auth=self.awsauth,
            use_ssl=True,
            verify_certs=True,
            connection_class=RequestsHttpConnection
        )
        
        # 2. Define the retriever (fetch top 3 most similar chunks)
        retriever = vector_store.as_retriever(search_kwargs={"k": 3})
        
        # 3. Create the prompt (Modern ChatPromptTemplate)
        system_prompt = (
            "You are a precise technical assistant. Answer the user's question using ONLY the provided context.\n"
            "If the answer is not in the context, say 'I do not know'.\n\n"
            "<context>\n{context}\n</context>"
        )
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", "{input}"),
        ])
        
        # 4. Construct LCEL Chains (Replaces legacy ConversationalRetrievalChain)
        question_answer_chain = create_stuff_documents_chain(self.llm, prompt)
        rag_chain = create_retrieval_chain(retriever, question_answer_chain)
        
        try:
            logger.info(f"Executing RAG chain for query: '{query}'")
            response = rag_chain.invoke({"input": query})
            return response["answer"]
        except Exception as e:
            logger.error(f"RAG query failed: {str(e)}")
            return "Error executing query."

if __name__ == "__main__":
    AOSS_ENDPOINT = "https://your-collection-id.us-east-1.aoss.amazonaws.com"
    INDEX_NAME = "bedrock-knowledge-index"
    
    manager = RAGPipelineManager(aoss_endpoint=AOSS_ENDPOINT)
    
    # Example execution (Assuming IAM and AOSS are configured)
    # manager.load_and_index_s3_document("my-training-data-bucket", "onboarding-guide.pdf", INDEX_NAME)
    # answer = manager.query_rag_pipeline("What is the standard architecture for the AI data pipeline?", INDEX_NAME)
    # print(answer)

```

---

## 5. Practice Challenges

### Challenge 1: Metadata Filtering in OpenSearch

**Scenario:** Your Vector Store contains HR documents for multiple departments. A user in Engineering asks a policy question, and the retriever incorrectly returns a policy meant for Sales because the semantic wording was extremely similar.

* **Task:** Modify the `as_retriever()` call to include a `search_kwargs={"filter": {"department": "engineering"}}`. You must also modify the ingestion step to append `{"department": "engineering"}` to the `metadata` dictionary of the LangChain `Document` object before inserting it into AOSS.

### Challenge 2: Transitioning to Amazon Kendra

**Scenario:** Managing text chunking sizes and OpenSearch embeddings manually is taking too much developer time. You decide to switch to Amazon Kendra, which natively crawls S3 buckets and indexes documents without custom embedding scripts.

* **Task:** Replace the `OpenSearchVectorSearch` implementation with `AmazonKendraRetriever` from `langchain_aws.retrievers`. Construct an LCEL chain that passes the `kendra_index_id` to the retriever and routes the results to `ChatBedrockConverse`.

---

## 6. Industry Standards & Tips

* **Amazon Kendra vs. OpenSearch Serverless:**
* Use **Kendra** when you want a fully managed, turn-key enterprise search. It natively connects to S3, SharePoint, and Confluence, handles OCR on PDFs, and manages its own embedding logic.
* Use **OpenSearch Serverless** when you need fine-grained control over the embedding models (e.g., you want to use the newest `amazon.titan-embed-text-v2:0`), custom text chunking strategies, or you are building complex multi-modal (image/text) similarity searches.


* **SigV4 Authentication is Mandatory:** When connecting LangChain to AWS managed databases (OpenSearch, Neptune, etc.), standard username/password authentication is insecure. Always use the `requests_aws4auth` library to sign HTTP requests natively with your IAM Role's short-lived STS credentials (SigV4).
* **Avoid the `ConversationalRetrievalChain`:** LangChain has formally deprecated older chain abstractions like `ConversationalRetrievalChain`. Modern AWS AI Engineering standardizes on LCEL (`create_retrieval_chain` + `create_stuff_documents_chain`) as it provides much cleaner debugging, native streaming support, and easier integration with memory.

---

## 7. Interview Q&A (Targeted for AI Engineers)

### Q1: Why do we convert text to vectors (embeddings) instead of just doing a direct SQL or Keyword search in a database?

> **Answer:** Keyword search relies on exact lexical matches. If a user asks "How do I terminate an employee?", a keyword search will miss a document titled "Offboarding Protocols" because the words don't match. Embeddings convert text into high-dimensional numerical vectors that represent *semantic meaning*. A vector database performs a mathematical cosine similarity search, allowing us to find conceptually relevant documents regardless of the specific vocabulary used.

### Q2: Explain the role of the LangChain Retriever component. Why not just query the database directly?

> **Answer:** The Retriever acts as an abstraction interface. Directly querying a vector database requires handling connection protocols, writing bespoke JSON payloads, generating embeddings for the query, and parsing raw database records. A LangChain Retriever standardizes all of this. It takes a raw string, silently embeds it, executes the query against the underlying store (OpenSearch, Kendra, or pgvector), and returns a standard list of LangChain `Document` objects that seamlessly pipe into the LLM.

### Q3: When loading documents from S3 into OpenSearch Serverless, what is the biggest risk of skipping the "Text Splitter" step?

> **Answer:** If we load a 100-page PDF from S3 and embed it as a single chunk, two critical failures occur. First, the resulting embedding vector will be extremely diluted, degrading the accuracy of the similarity search. Second, when the retriever fetches that massive document and injects it into the LLM's prompt, it will likely exceed the foundation model's context window token limit, causing the API call to fail. Documents must be chunked (e.g., 1000 characters) before embedding.